In [ ]:
Q. 1 — Perform a Small ETL Flow

In [2]:
import pandas as pd
 
source = pd.DataFrame({
    "product_id": [101, 102, 103, 104],
    "price_gbp": ["25.50", "invalid", "12.00", "40.00"],
})
 
working = source.copy()
 
working["price_gbp"] = pd.to_numeric(
    working["price_gbp"],
    errors="coerce",
)
 
clean_products = working[
    working["price_gbp"].notna()
].copy()
 
rejected_products = working[
    working["price_gbp"].isna()
].copy()
 
print("Clean rows:")
print(clean_products)
 
print("Rejected rows:")
print(rejected_products)

Clean rows:
   product_id  price_gbp
0         101       25.5
2         103       12.0
3         104       40.0
Rejected rows:
   product_id  price_gbp
1         102        NaN


In [ ]:
Q. 2 — Simulate a Small ELT Flow 

In [3]:
import pandas as pd
 
source = pd.DataFrame({
    "order_id": [201, 202, 203],
    "amount_gbp": ["45.50", "30.00", "bad"],
})
 
raw_orders = source.copy()
 
clean_orders = raw_orders.copy()
 
clean_orders["amount_gbp"] = pd.to_numeric(
    clean_orders["amount_gbp"],
    errors="coerce",
)
 
clean_orders = clean_orders[
    clean_orders["amount_gbp"].notna()
].copy()
 
print("Raw:")
print(raw_orders)
 
print("Clean:")
print(clean_orders)

Raw:
   order_id amount_gbp
0       201      45.50
1       202      30.00
2       203        bad
Clean:
   order_id  amount_gbp
0       201        45.5
1       202        30.0


In [ ]:
Q. 3 — Choose ETL or ELT 
Ans: 
Requirement	                            Choice	Reason
Remove invalid data before target	    ETL	    Transformation happens first
Keep raw data first	                    ELT	    Source data is loaded before transformation
Strong target SQL capability	        ELT	    Transformation can happen in the target
Remove sensitive fields before loading	ETL   	Sensitive values are changed before the target receives them

In [ ]:
Q. 4 — MCQ: ETL
Which sequence represents ETL?
Ans: C. Extract → Transform → Load

In [ ]:
Q. 5 — Process Three Independent Batches 

In [4]:
batches = {
    "batch_1": [10, 20, 30],
    "batch_2": [40, 50],
    "batch_3": [60, 70, 80],
}
 
for batch_name, values in batches.items():
    print(
        batch_name,
        "rows =", len(values),
        "total =", sum(values),
    )

batch_1 rows = 3 total = 60
batch_2 rows = 2 total = 90
batch_3 rows = 3 total = 210


In [ ]:
Q. 6 — Batch or Streaming?
Choose the better processing style. 
Ans: 
1. Supplier sends one file every night. -> Batch
2. Payment events arrive continuously and need a result within seconds. -> Streaming
3. Finance produces one monthly report. -> Batch
4. Website click events arrive throughout the day. -> Streaming
Batch handles collected groups of records. Streaming handles continuously arriving events when low delay matters.

In [ ]:
Q. 7 — Preserve Raw Input Before Cleaning

In [5]:
import pandas as pd
 
source = pd.DataFrame({
    "order_id": [301, 302, 303],
    "city": [" London ", None, "Leeds"],
    "amount_gbp": [45.50, 20.00, -5.00],
})
 
raw_orders = source.copy()
 
clean_orders = source[
    source["city"].notna()
    & (source["amount_gbp"] > 0)
].copy()
 
clean_orders["city"] = (
    clean_orders["city"].str.strip()
)
 
print("Raw rows:", len(raw_orders))
print("Clean rows:", len(clean_orders))
 
print(raw_orders)
print(clean_orders)

Raw rows: 3
Clean rows: 1
   order_id      city  amount_gbp
0       301   London         45.5
1       302       NaN        20.0
2       303     Leeds        -5.0
   order_id    city  amount_gbp
0       301  London        45.5


In [ ]:
Q. 8 — MCQ: Streaming
Which requirement is the strongest reason to use streaming?
Ans: A. Fraud events must be checked within seconds. 

In [ ]:
Q. 9 — Place Work in the Correct Layer
Classify each activity as mainly staging, transformation, or analytics. 
Ans: 
A. Rename source columns to standard names. -> Staging
B. Join orders to customers. -> Transformation
C. Calculate daily revenue by city. -> Analytics
D. Convert an amount column from text to numeric. -> Staging or transformation, depending on the design
E. Create a report-ready monthly sales table. -> Analytics

In [ ]:
Q. 10 — Build a Staging Output

In [6]:
import pandas as pd
 
source = pd.DataFrame({
    "OrderID": [401, 402, 403],
    "CustomerID": [1, 2, 1],
    "Amount": ["45.50", "30.00", "20.00"],
})
 
staging = source.rename(columns={
    "OrderID": "order_id",
    "CustomerID": "customer_id",
    "Amount": "amount_gbp",
}).copy()
 
staging["amount_gbp"] = pd.to_numeric(
    staging["amount_gbp"],
    errors="raise",
)
 
print(staging)

   order_id  customer_id  amount_gbp
0       401            1        45.5
1       402            2        30.0
2       403            1        20.0


In [ ]:
Q. 11 — Build an Analytics Output
Use this complete dataset:

In [7]:
import pandas as pd
 
orders = pd.DataFrame({
    "order_id": [501, 502, 503, 504],
    "city": ["London", "Leeds", "London", "Manchester"],
    "amount_gbp": [45.50, 30.00, 62.25, 50.00],
    "status": ["complete", "complete", "complete", "cancelled"],
})
 
completed = orders[
    orders["status"] == "complete"
].copy()
 
report = (
    completed
    .groupby("city", as_index=False)
    .agg(
        completed_revenue_gbp=(
            "amount_gbp",
            "sum",
        )
    )
)
 
print(report)

     city  completed_revenue_gbp
0   Leeds                  30.00
1  London                 107.75


In [ ]:
Q. 12 — MCQ: Staging
Which task is most suitable for a staging layer? 
Ans: D. Rename and standardise source columns before deeper transformation.

In [ ]:
Q. 13 — Put the Jobs in Order 
Ans: 
load_source
    ↓
clean_orders
    ↓
build_daily_sales
    ↓
publish_report

In [ ]:
Q. 14 — Retry or Fix First?
Choose whether an automatic retry is useful.
Ans:
Failure	                    Retry useful?	Reason
Temporary network timeout	Yes	            The condition may clear
Python syntax error        	No	            Code must be corrected
Short database outage	    Yes	            The service may recover
Required column removed	    No	            Input or code must be corrected


In [ ]:
Q. 15 — Plan a Backfill
Create a backfill plan.
Ans: A sensible plan is:

Confirm the source data exists for all three dates.
Check whether any partial output already exists.
Process 2026-08-05.
Validate counts and totals.
Process 2026-08-06.
Validate counts and totals.
Process 2026-08-07.
Validate counts and totals.
Confirm that rerunning a date will not duplicate data.

In [ ]:
Q. 16 — MCQ: Dependency
Why is a dependency needed between clean_orders and build_daily_sales? 
Ans: B. To ensure the sales summary waits for clean data.

In [ ]:
Q. 17 — Build a Complete Independent ETL Script

In [8]:
import pandas as pd
 
source = pd.DataFrame({
    "order_id": [601, 602, 603, 604],
    "city": ["London", None, "Leeds", "Manchester"],
    "amount_gbp": ["45.50", "30.00", "bad", "50.00"],
    "status": ["complete", "complete", "complete", "cancelled"],
})
 
raw_orders = source.copy()
 
working = raw_orders.copy()
 
working["amount_gbp"] = pd.to_numeric(
    working["amount_gbp"],
    errors="coerce",
)
 
clean_orders = working[
    working["city"].notna()
    & working["amount_gbp"].notna()
].copy()
 
completed = clean_orders[
    clean_orders["status"] == "complete"
].copy()
 
daily_sales = (
    completed
    .groupby("city", as_index=False)
    .agg(
        revenue_gbp=("amount_gbp", "sum")
    )
)
 
print("Raw:")
print(raw_orders)
 
print("Clean:")
print(clean_orders)
 
print("Daily sales:")
print(daily_sales)

Raw:
   order_id        city amount_gbp     status
0       601      London      45.50   complete
1       602         NaN      30.00   complete
2       603       Leeds        bad   complete
3       604  Manchester      50.00  cancelled
Clean:
   order_id        city  amount_gbp     status
0       601      London        45.5   complete
3       604  Manchester        50.0  cancelled
Daily sales:
     city  revenue_gbp
0  London         45.5


In [ ]:
Q. 18 — Find the Pipeline Design Problem 
Ans: The jobs are reversed. Each job needs output from the job before it.

Correct order:

load_source
    ↓
clean_orders
    ↓
build_daily_sales
    ↓
publish_report

In [ ]:
Q. 19 — Check Whether a Backfill Is Safe
Ans: The rerun can create duplicates because the script appends the same date again.
Before rerunning, the team should check:

whether output already exists for that date;
whether the job can replace or safely re-create that date;
whether a unique business key can identify duplicates;
whether counts and totals remain correct after the rerun.

In [ ]:
Q. 20 — Design a Four-Layer Batch Pipeline 
Ans: 
Layer	         Input	            Main work	                        Output	              Validation
Raw	             Daily order file	Keep received data	                Raw orders	          Input row count
Staging	         Raw orders	        Standardise columns and types	    Staged orders	      Required columns
Transformation	 Staged orders	    Clean, join, apply business rules	Trusted orders	      Valid/rejected counts
Analytics	     Trusted orders	    Aggregate for reporting	            Daily sales	          Revenue reconciliation
